In [92]:
import pandas as pd
import glob
import os

RAW_DIR = '../../data/norming_study/raw/b_pilot/'

In [93]:
# load all half csvs (skip demographics)
half_files = glob.glob(os.path.join(RAW_DIR, '*_half.csv'))
raw = pd.concat([pd.read_csv(f) for f in half_files], ignore_index=True)

print(f'{len(half_files)} files loaded, {len(raw)} total rows')
raw.head()

14 files loaded, 224 total rows


,trial_type,subjectID,prolificID,studyID,sessionID,DEBUG,sliderOrder,half,itemID,actionPhrase,...,abilityDragged,willingnessResponse,willingnessRT,willingnessDragged,conditionalMax,trialRT,trialIndex,suspicious,targetValue,passed
0,whyask-trial,0yhyx99mnk,NaN,NaN,NaN,0,WA,1,109,give me your honest take on my business idea,...,True,75.0,49016.0,True,75.0,54213.0,1,False,NaN,NaN
1,whyask-trial,0yhyx99mnk,NaN,NaN,NaN,0,WA,1,60,share your class notes with me,...,True,20.0,12610.0,True,20.0,35603.0,2,False,NaN,NaN
2,whyask-trial,0yhyx99mnk,NaN,NaN,NaN,0,WA,1,172,slaughter and clean the chicken for dinner,...,True,2.0,7607.0,True,2.0,14230.0,3,False,NaN,NaN
3,whyask-trial,0yhyx99mnk,NaN,NaN,NaN,0,WA,1,130,forge my supervisor's signature on my time sheet,...,True,5.0,10684.0,True,5.0,14456.0,4,False,NaN,NaN
4,whyask-trial,0yhyx99mnk,NaN,NaN,NaN,0,WA,1,141,let me stay in your apartment for one night,...,True,60.0,10548.0,True,60.0,13636.0,5,False,NaN,NaN


In [94]:
# split trials (attn checks removed from study)
trials = raw[raw['half'].isin([1, 2])].copy()
# attn = raw[raw['half'] == 'attn'].copy()

print(f'trials: {len(trials)} rows, {trials["subjectID"].nunique()} subjects')
print(f'subjects: {sorted(trials["subjectID"].unique())}')

trials: 105 rows, 7 subjects
subjects: ['0yhyx99mnk', '9jt5xmqcz1', 'bfvm0y4nso', 'fkexq8dztx', 'ozppteowo7', 'wvwpa4yudu', 'y0fe40cx3r']


In [95]:
# load demographics
demo_files = glob.glob(os.path.join(RAW_DIR, '*_demographics.csv'))
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

print(f'{len(demo)} demo rows')

7 demo rows


In [96]:
# attn check summary — commented out (no attn checks in study)
# attn_cols = ['subjectID', 'itemID', 'abilityResponse', 'willingnessResponse', 'suspicious']
# if 'targetValue' in attn.columns:
#     attn_cols.insert(4, 'targetValue')
# print(attn[attn_cols])

In [97]:
# attn check filtering commented out (no attn checks in study)
# failed = attn[attn['suspicious'] == True]['subjectID'].unique()
# trials_clean = trials[~trials['subjectID'].isin(failed)].copy()
# demo_clean   = demo[~demo['subjectID'].isin(failed)].copy()

trials_clean = trials.copy()
demo_clean   = demo.copy()

print(f'kept {trials_clean["subjectID"].nunique()} subjects ({len(trials_clean)} trials)')

kept 7 subjects (105 trials)


In [98]:
# quick look at trial responses
print(trials[['subjectID', 'itemID', 'actionPhrase', 'abilityResponse', 'willingnessResponse', 'sliderOrder']].to_string())
print()
# avg time per trial
avg_rt = trials['trialRT'].mean() / 1000
print(f'avg trial RT: {avg_rt:.1f}s')

      subjectID itemID                                                                        actionPhrase  abilityResponse  willingnessResponse sliderOrder
0    0yhyx99mnk    109                                        give me your honest take on my business idea            100.0                 75.0          WA
1    0yhyx99mnk     60                                                      share your class notes with me            100.0                 20.0          WA
2    0yhyx99mnk    172                                          slaughter and clean the chicken for dinner             70.0                  2.0          WA
3    0yhyx99mnk    130                                    forge my supervisor's signature on my time sheet            100.0                  5.0          WA
4    0yhyx99mnk    141                                         let me stay in your apartment for one night            100.0                 60.0          WA
5    0yhyx99mnk    170                                    

In [99]:
# save processed
PROC_DIR = '../../data/norming_study/processed/b_pilot/'
os.makedirs(PROC_DIR, exist_ok=True)

# drop trial-level cols from demo
trial_cols = ['trialIndex', 'itemID', 'actionPhrase', 'abilityResponse', 'abilityRT',
              'abilityDragged', 'willingnessResponse', 'willingnessRT', 'willingnessDragged',
              'trialRT', 'suspicious', 'targetValue', 'passed']
demo_out = demo_clean.drop(columns=[c for c in trial_cols if c in demo_clean.columns])

trials_clean.to_csv(os.path.join(PROC_DIR, 'trials.csv'), index=False)
demo_out.to_csv(os.path.join(PROC_DIR, 'demographics.csv'), index=False)

print(f'saved trials.csv ({len(trials_clean)} rows) and demographics.csv ({len(demo_out)} rows)')
print(f'demo cols: {list(demo_out.columns)}')

saved trials.csv (105 rows) and demographics.csv (7 rows)
demo cols: ['trial_type', 'subjectID', 'prolificID', 'studyID', 'sessionID', 'DEBUG', 'sliderOrder', 'half', 'age', 'gender', 'race', 'education', 'strategy', 'technicalIssues', 'feedback', 'visibilityChanges', 'totalDurationMs']


In [100]:
import re, json

# all itemIDs w/ responses across every processed pilot
all_trials = pd.concat([
    pd.read_csv(f) for f in glob.glob('../../data/norming_study/processed/*/trials.csv')
], ignore_index=True)
seen = set(all_trials['itemID'].dropna().astype(int))

# parse stimuli.js
with open('../../experiments/norming_study/stimuli.js') as f:
    raw_js = f.read()
stimuli_data = json.loads(re.sub(r'^var STIMULI_DATA\s*=\s*', '', raw_js.strip().rstrip(';')))

unseen = [s for s in stimuli_data['stimuli'] if s['itemID'] not in seen]

# write stimuli_not_seen.js (overwritten each run)
lines = [
    'var STIMULI_DATA = {',
    '  "experimentName": "whyask_norming",',
    '  "version": "v4_unseen",',
    f'  "nItems": {len(unseen)},',
    '  "stimuli": [',
]
for i, s in enumerate(unseen):
    comma = '' if i == len(unseen) - 1 else ','
    lines += [
        '    {',
        f'      "itemID": {s["itemID"]},',
        f'      "actionPhrase": "{s["actionPhrase"].replace(chr(34), chr(92)+chr(34))}",',
        f'      "vignette": "{s["vignette"].replace(chr(34), chr(92)+chr(34))}"',
        f'    }}{comma}',
    ]
lines += ['  ]', '};']

with open('../../experiments/norming_study/stimuli_not_seen.js', 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(f'{len(seen)} items seen, {len(unseen)} unseen → wrote stimuli_not_seen.js')

122 items seen, 55 unseen → wrote stimuli_not_seen.js
